**Feature Selection Method:** PSO   
**Dataset:** QUT-DV25 (TCP)   
**Developed By:** eDySec Research Team   
**Plartform:** Ubuntu

All experiments in this notebook were conducted using **Python 3.10** with the following libraries:

`pandas==1.5.3`,  
`scikit-learn==1.2.2`,  
`openpyxl`,  
`numpy==1.23.5`,  
`scipy==1.9.3`,  
`tensorflow==2.11.0`,  
`matplotlib==3.7.1`,  
`seaborn==0.12.2`,  
`joblib==1.3.2`,  
`shap==0.41.0`,  
`lime`,  
`flaml[automl]==2.5.0`,  
`notebook==6.5.6`,  
`pywinpty==2.0.10`  (Only for windows)  `threadpoolctl==3.1.0` (for Ubuntu)   
`terminado==0.17.1`,  
`transformers==4.49.0`.

#### Full Environment Setup: https://github.com/tanzirmehedi/eDySec

These versions were used to ensure **consistent and reproducible experimental results**.

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from scipy.sparse import hstack
from scipy.sparse import csr_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, roc_auc_score, roc_curve,
    precision_recall_fscore_support, cohen_kappa_score
)
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
import time
import joblib
import gc
from sklearn.feature_selection import f_classif
from flaml import AutoML

In [3]:
# This will allow TensorFlow to allocate as much GPU memory as needed, including full 16GB if needed.
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # Disable memory growth (optional - default is False)
        tf.config.experimental.set_memory_growth(gpus[0], False)
        print("Memory growth disabled (default behavior).")
    except RuntimeError as e:
        print("Error:", e)

Memory growth disabled (default behavior).


In [ ]:
file_path = 'QUT-DV25_'+DATASET_NAME+'_Traces.csv'
data = pd.read_csv(file_path)

In [4]:
gc.collect()
tf.keras.backend.clear_session()

In [4]:
data.head()

,Package_Name,Total_Entries,Unique_C-COMM,Python_Related_Process,State_Transition,Local_IP_Address_Access,Remote_IP_Address_Access,Local_Port_Access,Remote_Port_Access,Level
0,10Cent10-999.0.4.tar.gz,946,7,316,"{'FIN_WAIT1 -> ->': 188, 'SYN_SENT -> ->': 187...",2,62,194,21,1
1,10Cent11-999.0.4.tar.gz,874,6,301,"{'SYN_SENT -> ->': 177, 'FIN_WAIT1 -> ->': 170...",2,51,181,18,1
2,11Cent-999.0.0.tar.gz,801,4,272,"{'FIN_WAIT1 -> ->': 160, 'SYN_SENT -> ->': 157...",2,52,164,17,1
3,11Cent-999.0.1.tar.gz,728,4,254,"{'SYN_SENT -> ->': 149, 'FIN_WAIT1 -> ->': 141...",2,39,152,14,1
4,11Cent-999.0.2.tar.gz,655,4,234,"{'SYN_SENT -> ->': 133, 'FIN_WAIT1 -> ->': 128...",2,32,135,11,1


In [8]:
X_df = data.drop(columns=['Level']).copy()
y = data['Level']

# Encode label if needed
if y.dtype == 'object' or y.dtype.name == 'category':
    y = LabelEncoder().fit_transform(y)

# Identify categorical columns (strings)
cat_cols = X_df.select_dtypes(include='object').columns
num_cols = X_df.select_dtypes(include=['int64','float64']).columns

# Simple Label Encoding for categorical columns
for col in cat_cols:
    X_df[col] = LabelEncoder().fit_transform(X_df[col].astype(str))

# Now all columns are numeric
X = X_df.values
feature_names = list(X_df.columns)

# =====================
# Fitness Function
# =====================
def fitness_function(selected_features, X_train, y_train, X_val, y_val):
    if np.sum(selected_features) == 0:
        return 0
    selected_cols = np.where(selected_features == 1)[0]
    model = LogisticRegression(max_iter=500)
    model.fit(X_train[:, selected_cols], y_train)
    preds = model.predict(X_val[:, selected_cols])
    return accuracy_score(y_val, preds)

# =====================
# PSO Feature Selection (20 iterations)
# =====================
def pso_feature_selection(X, y, num_particles=5, max_iter=20):
    n_features = X.shape[1]
    particles = np.random.randint(2, size=(num_particles, n_features))
    velocities = np.random.uniform(-1, 1, (num_particles, n_features))

    pbest_positions = particles.copy()
    pbest_scores = np.zeros(num_particles)
    gbest_position = None
    gbest_score = -1

    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # Initialize pbest and gbest
    for i in range(num_particles):
        score = fitness_function(particles[i], X_train, y_train, X_val, y_val)
        pbest_scores[i] = score
        if score > gbest_score:
            gbest_score = score
            gbest_position = particles[i].copy()

    w, c1, c2 = 0.7, 1.5, 1.5
    for t in range(max_iter):
        for i in range(num_particles):
            r1, r2 = np.random.rand(), np.random.rand()
            velocities[i] = (
                w * velocities[i]
                + c1 * r1 * (pbest_positions[i] - particles[i])
                + c2 * r2 * (gbest_position - particles[i])
            )
            sigmoid = 1 / (1 + np.exp(-velocities[i]))
            particles[i] = (np.random.rand(n_features) < sigmoid).astype(int)

            score = fitness_function(particles[i], X_train, y_train, X_val, y_val)
            if score > pbest_scores[i]:
                pbest_scores[i] = score
                pbest_positions[i] = particles[i].copy()
                if score > gbest_score:
                    gbest_score = score
                    gbest_position = particles[i].copy()

        selected = [feature_names[i] for i in range(n_features) if gbest_position[i]==1]
        dropped = [feature_names[i] for i in range(n_features) if gbest_position[i]==0]
        print(f"Iteration {t+1}/{max_iter}, Best Accuracy: {gbest_score:.4f}")
        print(f"Selected features: {selected}")
        print(f"Dropped features: {dropped}\n")

    # Final features after all iterations
    final_selected = [feature_names[i] for i in range(n_features) if gbest_position[i]==1]
    final_dropped = [feature_names[i] for i in range(n_features) if gbest_position[i]==0]
    print("===== FINAL RESULTS AFTER 20 ITERATIONS =====")
    print(f"Best Accuracy: {gbest_score:.4f}")
    print(f"Final Selected features: {final_selected}")
    print(f"Final Dropped features: {final_dropped}")

    return gbest_position, gbest_score

best_features, best_score = pso_feature_selection(X, y, num_particles=5, max_iter=20)


Iteration 1/20, Best Accuracy: 0.7310
Selected features: ['Package_Name', 'Total_Entries', 'Unique_C-COMM', 'Python_Related_Process', 'State_Transition', 'Remote_IP_Address_Access', 'Local_Port_Access', 'Remote_Port_Access']
Dropped features: ['Local_IP_Address_Access']

Iteration 2/20, Best Accuracy: 0.7471
Selected features: ['Total_Entries', 'Python_Related_Process', 'Local_IP_Address_Access', 'Remote_IP_Address_Access', 'Remote_Port_Access']
Dropped features: ['Package_Name', 'Unique_C-COMM', 'State_Transition', 'Local_Port_Access']



C:\Users\USER\anaconda3\envs\EEE\lib\site-packages\sklearn\linear_model\_logistic.py:818: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  extra_warning_msg=_LOGISTIC_SOLVER_CONVERGENCE_MSG,


Iteration 3/20, Best Accuracy: 0.7471
Selected features: ['Total_Entries', 'Python_Related_Process', 'Local_IP_Address_Access', 'Remote_IP_Address_Access', 'Remote_Port_Access']
Dropped features: ['Package_Name', 'Unique_C-COMM', 'State_Transition', 'Local_Port_Access']

Iteration 4/20, Best Accuracy: 0.7471
Selected features: ['Total_Entries', 'Python_Related_Process', 'Local_IP_Address_Access', 'Remote_IP_Address_Access', 'Remote_Port_Access']
Dropped features: ['Package_Name', 'Unique_C-COMM', 'State_Transition', 'Local_Port_Access']

Iteration 5/20, Best Accuracy: 0.7538
Selected features: ['Total_Entries', 'Unique_C-COMM', 'Python_Related_Process', 'Local_IP_Address_Access', 'Remote_IP_Address_Access', 'Remote_Port_Access']
Dropped features: ['Package_Name', 'State_Transition', 'Local_Port_Access']

Iteration 6/20, Best Accuracy: 0.7538
Selected features: ['Total_Entries', 'Unique_C-COMM', 'Python_Related_Process', 'Local_IP_Address_Access', 'Remote_IP_Address_Access', 'Remote_Por

In [9]:
selected_features = ['Total_Entries', 'Python_Related_Process', 'Local_IP_Address_Access', 'Local_Port_Access', 'Remote_Port_Access']